In [175]:
years = list(range(1991,2026))

In [154]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
options = Options()
options.add_argument("--disable-gpu")
options.add_argument("--disable-extensions")
options.add_argument("--blink-settings=imagesEnabled=false")

service = Service(executable_path="/Users/aidenkrueger/chromedriver-mac-x64/chromedriver")
driver = webdriver.Chrome(service=service)

In [12]:
url_start = "https://www.basketball-reference.com/awards/awards_{}.html"

In [156]:
import requests
import time
import cloudscraper

scraper = cloudscraper.create_scraper()

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/122.0.0.0 Safari/527.36"
}

for year in years:
    url = url_start.format(year)
    time.sleep(1)
    driver.get(url)

    with open("nba_mvp/mvp/{}.html".format(year),"w+") as f:
    f.write(driver.page_source)


KeyboardInterrupt: 

In [25]:
from bs4 import BeautifulSoup

In [176]:
import pandas as pd
dfs = []
for year in years:
    with open("nba_mvp/mvp/{}.html".format(year)) as f:
        page = f.read()
    soup = BeautifulSoup(page, 'html.parser')
    soup.find('tr',class_="over_header").decompose()
    mvp_table = soup.find(id="mvp")
    mvp = pd.read_html(str(mvp_table))[0]
    mvp["Year"] = year
    
    dfs.append(mvp)

/var/folders/8_/w_wrwrbs3r7fmmvrx9gmk_000000gn/T/ipykernel_1527/3866965389.py:9: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  mvp = pd.read_html(str(mvp_table))[0]
/var/folders/8_/w_wrwrbs3r7fmmvrx9gmk_000000gn/T/ipykernel_1527/3866965389.py:9: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  mvp = pd.read_html(str(mvp_table))[0]
/var/folders/8_/w_wrwrbs3r7fmmvrx9gmk_000000gn/T/ipykernel_1527/3866965389.py:9: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  mvp = pd.read_html(str(mvp_table))[0]
/var/folders/8_/w_wrwrbs3r7fmmvrx9gmk_000000gn/T/ipykernel_1527/3866965389.py:9: FutureWarning: Passing literal html to 'read_html

In [177]:
mvps = pd.concat(dfs)

In [178]:
mvps.tail(20)

,Rank,Player,Age,Tm,First,Pts Won,Pts Max,Share,G,MP,...,TRB,AST,STL,BLK,FG%,3P%,FT%,WS,WS/48,Year
1,2,Shai Gilgeous-Alexander,25,OKC,15,640,990,0.646,75,34.0,...,5.5,6.2,2.0,0.9,0.535,0.353,0.874,14.6,0.275,2024
2,3,Luka Dončić,24,DAL,4,566,990,0.572,70,37.5,...,9.2,9.8,1.4,0.5,0.487,0.382,0.786,12.0,0.220,2024
3,4,Giannis Antetokounmpo,29,MIL,1,192,990,0.194,73,35.2,...,11.5,6.5,1.2,1.1,0.611,0.274,0.657,13.2,0.246,2024
4,5,Jalen Brunson,27,NYK,0,142,990,0.143,77,35.4,...,3.6,6.7,0.9,0.2,0.479,0.401,0.847,11.2,0.198,2024
5,6,Jayson Tatum,25,BOS,0,86,990,0.087,74,35.7,...,8.1,4.9,1.0,0.6,0.471,0.376,0.833,10.4,0.189,2024
6,7,Anthony Edwards,22,MIN,0,18,990,0.018,79,35.1,...,5.4,5.1,1.3,0.5,0.461,0.357,0.836,7.5,0.130,2024
7,8,Domantas Sabonis,27,SAC,0,3,990,0.003,82,35.7,...,13.7,8.2,0.9,0.6,0.594,0.379,0.704,12.6,0.206,2024
8,9,Kevin Durant,35,PHO,0,1,990,0.001,75,37.2,...,6.6,5.0,0.9,1.2,0.523,0.413,0.856,8.3,0.142,2024
0,1,Shai Gilgeous-Alexander,26,OKC,71,913,1000,0.913,76,34.2,...,5.0,6.4,1.7,1.0,0.519,0.375,0.898,16.7,0.309,2025
1,2,Nikola Jokić,29,DEN,29,787,1000,0.787,70,36.7,...,12.7,10.2,1.8,0.6,0.576,0.417,0.800,16.4,0.307,2025


In [162]:
mvps.to_csv("mvps.csv")

In [62]:
player_stats_url = "https://www.basketball-reference.com/leagues/NBA_{}_per_game.html"

In [106]:
service = Service(executable_path="/Users/aidenkrueger/chromedriver-mac-x64/chromedriver")
driver = webdriver.Chrome(service=service)

In [107]:
for year in years:
    url = player_stats_url.format(year)

    driver.get(url)
    driver.execute_script("window.scrollTo(1,10000)")
    time.sleep(5)

    with open("nba_mvp/player/{}.html".format(year), "w+") as f:
        f.write(driver.page_source)
    

In [180]:
dfs = []
for year in years:
    with open("nba_mvp/player/{}.html".format(year)) as f:
        page = f.read()
    
    soup = BeautifulSoup(page, 'html.parser')
    for tr in soup.find_all('tr', class_=["norank", "thead"]):
        tr.decompose()
    player_table = soup.find(id="per_game_stats")
    player = pd.read_html(str(player_table))[0]
    player["Year"] = year
    
    dfs.append(player)

/var/folders/8_/w_wrwrbs3r7fmmvrx9gmk_000000gn/T/ipykernel_1527/866565754.py:10: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  player = pd.read_html(str(player_table))[0]
/var/folders/8_/w_wrwrbs3r7fmmvrx9gmk_000000gn/T/ipykernel_1527/866565754.py:10: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  player = pd.read_html(str(player_table))[0]
/var/folders/8_/w_wrwrbs3r7fmmvrx9gmk_000000gn/T/ipykernel_1527/866565754.py:10: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  player = pd.read_html(str(player_table))[0]
/var/folders/8_/w_wrwrbs3r7fmmvrx9gmk_000000gn/T/ipykernel_1527/866565754.py:10: FutureWarning: Passing literal 

In [181]:
players = pd.concat(dfs)

In [182]:
players

,Rk,Player,Age,Team,Pos,G,GS,MP,FG,FGA,...,DRB,TRB,AST,STL,BLK,TOV,PF,PTS,Awards,Year
0,1,Michael Jordan,27,CHI,SG,82,82,37.0,12.1,22.4,...,4.6,6.0,5.5,2.7,1.0,2.5,2.8,31.5,"MVP-1,DPOY-7,AS,NBA1,DEF1",1991
1,2,Karl Malone,27,UTA,PF,82,82,40.3,10.3,19.6,...,8.9,11.8,3.3,1.1,1.0,3.0,3.3,29.0,"MVP-5,AS,NBA1",1991
2,3,Bernard King,34,WSB,SF,64,64,37.5,11.1,23.6,...,3.2,5.0,4.6,0.9,0.3,4.0,2.9,28.4,"MVP-16,AS,NBA3",1991
3,4,Charles Barkley,27,PHI,SF,67,67,37.3,9.9,17.4,...,6.3,10.1,4.2,1.6,0.5,3.1,2.6,27.6,"MVP-4,AS,NBA1",1991
4,5,Patrick Ewing,28,NYK,C,81,81,38.3,10.4,20.3,...,8.8,11.2,3.0,1.0,3.2,3.6,3.5,26.6,"MVP-11,DPOY-7,AS,NBA2",1991
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
730,565,Riley Minix,24,SAS,SF,1,0,7.0,0.0,1.0,...,2.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,2025
731,566,Jahlil Okafor,29,IND,C,1,0,3.0,0.0,0.0,...,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,NaN,2025
732,567,Zyon Pullin,23,MEM,SG,3,0,1.0,0.0,0.3,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,2025
733,568,Isaiah Stevens,24,MIA,PG,3,0,2.0,0.0,0.7,...,0.7,0.7,0.0,0.3,0.0,0.0,0.0,0.0,NaN,2025


In [183]:
players.to_csv("players.csv")

In [112]:
team_stats_url = "https://www.basketball-reference.com/leagues/NBA_{}_standings.html"

In [119]:
scraper = cloudscraper.create_scraper()

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/122.0.0.0 Safari/527.36"
}

for year in years:
    url = team_stats_url.format(year)
    data = scraper.get(url)
    time.sleep(2)
    with open("nba_mvp/team/{}.html".format(year), "w+") as f:
        f.write(data.text)

In [130]:
dfs = []
for year in years:
    with open("nba_mvp/team/{}.html".format(year)) as f:
        page = f.read()
        
    soup = BeautifulSoup(page, "html.parser")
    soup.find('tr',class_="thead").decompose()
    team_table = soup.find(id="divs_standings_E")
    team = pd.read_html(str(team_table))[0]
    team["Year"] = year
    team["Team"] = team["Eastern Conference"]
    del team["Eastern Conference"]
    dfs.append(team)

    soup = BeautifulSoup(page, "html.parser")
    soup.find('tr',class_="thead").decompose()
    team_table = soup.find(id="divs_standings_W")
    team = pd.read_html(str(team_table))[0]
    team["Year"] = year
    team["Team"] = team["Western Conference"]
    del team["Western Conference"]
    dfs.append(team)

/var/folders/8_/w_wrwrbs3r7fmmvrx9gmk_000000gn/T/ipykernel_1527/611525228.py:9: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  team = pd.read_html(str(team_table))[0]
/var/folders/8_/w_wrwrbs3r7fmmvrx9gmk_000000gn/T/ipykernel_1527/611525228.py:18: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  team = pd.read_html(str(team_table))[0]
/var/folders/8_/w_wrwrbs3r7fmmvrx9gmk_000000gn/T/ipykernel_1527/611525228.py:9: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  team = pd.read_html(str(team_table))[0]
/var/folders/8_/w_wrwrbs3r7fmmvrx9gmk_000000gn/T/ipykernel_1527/611525228.py:18: FutureWarning: Passing literal html to 'read_

In [131]:
teams = pd.concat(dfs)

In [132]:
teams

,W,L,W/L%,GB,PS/G,PA/G,SRS,Year,Team
0,56,26,.683,—,111.5,105.7,5.22,1991,Boston Celtics*
1,44,38,.537,12.0,105.4,105.6,-0.39,1991,Philadelphia 76ers*
2,39,43,.476,17.0,103.1,103.3,-0.43,1991,New York Knicks*
3,30,52,.366,26.0,101.4,106.4,-4.84,1991,Washington Bullets
4,26,56,.317,30.0,102.9,107.5,-4.53,1991,New Jersey Nets
...,...,...,...,...,...,...,...,...,...
13,52,30,.634,—,114.3,109.8,4.97,2025,Houston Rockets*
14,48,34,.585,4.0,121.7,116.9,4.79,2025,Memphis Grizzlies*
15,39,43,.476,13.0,114.2,115.4,-0.74,2025,Dallas Mavericks
16,34,48,.415,18.0,113.9,116.7,-2.45,2025,San Antonio Spurs


In [133]:
teams.to_csv("teams.csv")